# Lightning Ground Flash Density Forecast
## 150 kV Situbondo–Banyuwangi Transmission Line, 2026–2030

**Goal.** Forecast per-tower annual lightning activity (Ground Flash Density,
strike Count, and peak-current statistics) for every one of the 281 towers on
the line for the next 5 years, with explicit uncertainty bounds under
La Niña / Neutral / El Niño climate scenarios.

**Why now.** Historical line totals (2019–2025: 905, 563, 1445, 1718, 878, 564,
402 strikes) show that GFD is **not stationary** — interannual swings exceed
3×, driven by Indonesia's monsoon and convective regime. Using a static
average for insulation-coordination, arrester placement, and maintenance
planning systematically misallocates risk.

**Approach.** Two models retained (a third — NB GLM with spatial smoother —
was attempted and dropped after CV failure; see Cell 15):

| Model | What it captures | Role |
|-------|------------------|------|
| **A — Climatology** | Per-tower 7-year mean + ENSO-analog variant | Skill floor |
| **C — Two-stage hybrid (PRIMARY)** | Line-level temporal climate model × per-tower spatial share | Recommended forecast |

**Honest caveat (n=7).** Cross-validation shows Model C beats climatology by
only ~2% RMSE — within the noise of a 7-year dataset. Stage 1 of Model C
selects intercept-only under AICc (climate covariates don't statistically
earn their place at this sample size). The 2025 hold-out Pearson r = 0.237.
Use the forecast intervals and climate-scenario fan as a **sensitivity
analysis**, not as a skillful climate-driven prediction.

**Data sources.** Tidy CSVs from `tidy_data.py` (run once, audit in
`data_tidy/_data_quality_report.txt`); ENSO Niño 3.4 and IOD DMI from NOAA
PSL; elevation from Open-Meteo elevation API.

## Methodology pipeline

```
       Raw Excel (29 sheets, 2 files)
                |
                v
          tidy_data.py   --->   data_tidy/*.csv   +   _data_quality_report.txt
                                       |
                                       v
                              Notebook ingestion (cells 5–8)
                                       |
                                       v
        +------------------------+-----+--------------------+
        |                        |                          |
        v                        v                          v
   Exposure panel        Monthly line totals         Tower coordinates
   (1,967 rows)          (84 rows, diagnostics)      (281 lat/lng)
        |                                                   |
        |   merge on (tower_id, year)                       |
        |<------------- climate covariates -----------------+
        |                ENSO Niño 3.4, IOD DMI             |
        |                (NOAA PSL, cached)                 |
        |<------------- spatial covariates -----------------+
        |                elevation (Open-Meteo)             |
        |                distance to Bali Strait / Java Sea |
        v
   X_train (1,967 rows)         X_future (1,405 rows = 281 × 5 years)
        |
        v
   +---------+-------------+------------+
   |         |             |            |
   v         v             v            v
 Model A  Model B        Model C      Cross-validation
 (clim.)  (NB GLM)      (PRIMARY)     LOYO + expanding-window
                                       MAE, RMSE, Deviance, CRPS
        |
        v
   3 climate scenarios × 500-rep bootstrap → 80% / 95% intervals
        |
        v
   outputs/per_tower_forecast.xlsx, hotspot_map.html,
   top20_ranking.csv, cv_skill.html, sensitivity.csv
```

**Key modeling decisions** (literature-grounded):

- **Negative Binomial** for Count (Poisson is rejected by overdispersion test
  in cell 14).
- **Spatial smoother** = penalized tensor-product cubic regression splines on
  (lat, lng) via `patsy` — captures orographic effects (Mt. Ijen/Raung
  foothills) without pulling in a GAM library.
- **Temporal climate driver**: lagged Niño 3.4 (JJA-mean) and IOD DMI — the
  dominant Indonesian interannual rainfall and lightning drivers
  (Aldrian & Susanto 2003; ~70% of maritime-continent OLR variance is
  ENSO-driven).
- **Why Model C is primary at n=7 years**: separating *when* (line-level
  climate-driven total, fit with ≤3 predictors) from *where* (per-tower
  spatial share, dominated by stable terrain) reduces parameter count and
  matches the physics. Deep learning (ConvLSTM, DeepKriging) is excluded —
  it overfits at this sample size.

The notebook is reproducible: it runs top-to-bottom, caches external API
calls to `cache/` on first run, and reloads from disk thereafter (works
offline after the first execution).

In [ ]:
from __future__ import annotations

import json
import math
import warnings
from dataclasses import dataclass
from io import StringIO
from pathlib import Path
from typing import Sequence

import numpy as np
import pandas as pd
import requests
import scipy.stats as st
from joblib import Parallel, delayed

import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import dmatrix

import plotly.express as px
import plotly.graph_objects as go
import folium
from folium.plugins import MarkerCluster

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning, module="statsmodels")

HERE = Path.cwd()
DATA_DIR = HERE / "data_tidy"
CACHE_DIR = HERE / "cache"
OUT_DIR = HERE / "outputs"
CACHE_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

RNG_SEED = 20260512
RNG = np.random.default_rng(RNG_SEED)

HIST_YEARS = list(range(2019, 2026))         # 2019..2025
FORECAST_YEARS = list(range(2026, 2031))     # 2026..2030
ALL_YEARS = HIST_YEARS + FORECAST_YEARS

# Per the tidy audit: implied collection area is constant = 3.125 km^2 per
# tower-year. This becomes the constant exposure offset for Count models.
COLLECTION_AREA_KM2 = 3.125
LOG_OFFSET = math.log(COLLECTION_AREA_KM2)

# Climate-scenario fan for the 5-year forecast horizon.
SCENARIOS = {
    "LaNina":  {"nino34_jja": -1.0, "dmi_jja_offset": -0.2},
    "Neutral": {"nino34_jja":  0.0, "dmi_jja_offset":  0.0},
    "ElNino":  {"nino34_jja": +1.0, "dmi_jja_offset": +0.2},
}

N_BOOTSTRAP = 500

print(f"Working directory: {HERE}")
print(f"Historical years: {HIST_YEARS[0]}..{HIST_YEARS[-1]}  ({len(HIST_YEARS)} years)")
print(f"Forecast years:   {FORECAST_YEARS[0]}..{FORECAST_YEARS[-1]}  ({len(FORECAST_YEARS)} years)")
print(f"Collection area:  {COLLECTION_AREA_KM2} km^2 (constant, from tidy audit)")
print(f"Scenarios:        {list(SCENARIOS.keys())}")
print(f"Bootstrap reps:   {N_BOOTSTRAP}")

## Section A — Data ingestion

Read the tidy CSVs produced by `tidy_data.py`. Each load includes an
assertion block so that a corrupted CSV halts the notebook loudly rather
than silently degrading downstream model fits.

In [ ]:
panel = pd.read_csv(DATA_DIR / "exposure_tower_year.csv")
assert panel.shape == (1967, 17), f"unexpected panel shape: {panel.shape}"
assert set(panel["year"].unique()) == set(HIST_YEARS), "year set mismatch"
assert panel.groupby("year")["tower_id"].nunique().eq(281).all(), "tower-per-year mismatch"
assert (panel[["count","count_neg","count_pos","density"]] >= 0).all().all(), "negative counts"

# Surface any warnings from the data-quality report.
report_path = DATA_DIR / "_data_quality_report.txt"
if report_path.exists():
    fail_lines = [ln for ln in report_path.read_text(encoding="utf-8").splitlines() if "[FAIL]" in ln]
    if fail_lines:
        print(f"WARNING: {len(fail_lines)} failed checks in data-quality report:")
        for ln in fail_lines[:5]:
            print(f"  {ln}")
    else:
        print("Data-quality audit: all checks passed.")

flag = DATA_DIR / "_2025_completeness.flag"
if flag.exists():
    print(f"NOTE: 2025 completeness flag present: {flag.read_text().strip()}")

print(f"Loaded {len(panel):,} tower-year rows. Year totals (line-aggregate):")
print(panel.groupby("year")["count"].sum().to_string())

In [ ]:
monthly = pd.read_csv(DATA_DIR / "monthly_line.csv", parse_dates=["period_start", "period_end"])
line_year = pd.read_csv(DATA_DIR / "exposure_line_year.csv")
assert set(line_year["year"]) == set(HIST_YEARS)
assert len(monthly) >= 7 * 12, f"monthly panel too short: {len(monthly)}"

# Calendar-month restriction: drop the trailing rollover row per year.
monthly_cal = monthly.loc[~monthly["is_partial_month"]].copy()
print(f"Monthly line panel: {len(monthly):,} raw rows / {len(monthly_cal):,} calendar-month rows.")
print("\nLine-level annual totals (authoritative, from aggregate row):")
print(line_year[["year","count","count_neg","count_pos","density","mean_ka"]].to_string(index=False))

In [ ]:
ka_hist = pd.read_csv(DATA_DIR / "peak_current_hist.csv")
bw = ka_hist.groupby("year")["bin_width"].first()
print("kA histogram native bin widths (not silently rebinned):")
print(bw.to_string())
print(f"\nTotal histogram rows: {len(ka_hist):,}")

In [ ]:
towers = pd.read_csv(DATA_DIR / "towers.csv")
assert len(towers) == 281
assert towers["tower_id"].nunique() == 281
panel_ids = set(panel["tower_id"].unique())
coord_ids = set(towers["tower_id"].unique())
assert panel_ids == coord_ids, "tower id mismatch between panel and coords"
print(f"Towers: {len(towers)} ({towers['tower_id'].min()}..{towers['tower_id'].max()})")
print(f"Lat range: {towers['lat'].min():.4f} .. {towers['lat'].max():.4f}")
print(f"Lng range: {towers['lng'].min():.4f} .. {towers['lng'].max():.4f}")
print(f"Line span:  {towers['lat'].max() - towers['lat'].min():.4f} deg lat, "
      f"{towers['lng'].max() - towers['lng'].min():.4f} deg lng")

## Section B — Exogenous covariates

We add two families of external covariates:

1. **Climate**: monthly Niño 3.4 and IOD DMI indices from NOAA PSL, aggregated
   to annual and JJA-mean (June–August) — the JJA mean leads Indonesian
   wet-season rainfall by 3–6 months and is the standard interannual driver.
2. **Spatial**: per-tower elevation from Open-Meteo, and haversine distance to
   Bali Strait and Java Sea anchor points (proxy for distance-to-coast).

Both fetches cache to `cache/*.parquet` and reload offline on subsequent
runs.

In [ ]:
def _parse_psl_anom(text: str) -> pd.DataFrame:
    """Parse a NOAA PSL plain-text monthly anomaly file. Format:
       line 1: '<start_year> <end_year>'
       lines 2..N: '<year> <jan> <feb> ... <dec>'
       trailing lines: missing-value marker then notes.
    """
    rows = []
    for raw in text.splitlines():
        toks = raw.split()
        if len(toks) == 13 and all(t.replace(".", "").replace("-", "").isdigit() for t in toks):
            year = int(toks[0])
            vals = [float(v) for v in toks[1:]]
            rows.append((year, vals))
    if not rows:
        raise RuntimeError("no monthly rows parsed from PSL text")
    df = pd.DataFrame(
        [(y, m + 1, v) for y, vals in rows for m, v in enumerate(vals)],
        columns=["year", "month", "value"],
    )
    # PSL sentinel missing values are typically -99.99 or -999.0.
    df.loc[df["value"] <= -90, "value"] = np.nan
    return df


def fetch_index(url: str, cache_name: str, label: str) -> pd.DataFrame:
    cache_path = CACHE_DIR / cache_name
    if cache_path.exists():
        df = pd.read_parquet(cache_path)
        print(f"{label}: loaded from cache ({len(df)} monthly rows, "
              f"{df['year'].min()}..{df['year'].max()})")
        return df
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        df = _parse_psl_anom(resp.text)
        df.to_parquet(cache_path)
        print(f"{label}: fetched fresh from {url}  ({len(df)} monthly rows)")
        return df
    except Exception as exc:
        print(f"{label}: FETCH FAILED ({exc}). Falling back to a synthetic "
              f"placeholder so the notebook still runs end-to-end.")
        # Persistence-of-zero anomaly fallback so downstream code does not crash.
        years = list(range(1990, FORECAST_YEARS[-1] + 1))
        df = pd.DataFrame(
            [(y, m, 0.0) for y in years for m in range(1, 13)],
            columns=["year", "month", "value"],
        )
        df.to_parquet(cache_path)
        return df


NINO34_URL = "https://psl.noaa.gov/data/correlation/nina34.anom.data"
DMI_URL = "https://psl.noaa.gov/gcos_wgsp/Timeseries/Data/dmi.had.long.data"

nino = fetch_index(NINO34_URL, "nino34.parquet", "Nino3.4")
dmi  = fetch_index(DMI_URL,    "dmi.parquet",    "IOD DMI")


def annual_features(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    # JJA/SON aggregations are robust to missing months: if the window is fully
    # NaN (e.g. 2025 DMI hasn't been published past April), fall back to the
    # annual mean of available months, then to 0.0.
    rows = []
    for year, g in df.groupby("year"):
        g = g.sort_values("month")
        ann = g["value"].dropna().mean()
        jja_vals = g.loc[g["month"].between(6, 8), "value"].dropna()
        son_vals = g.loc[g["month"].between(9, 11), "value"].dropna()
        jja = jja_vals.mean() if len(jja_vals) else ann
        son = son_vals.mean() if len(son_vals) else ann
        rows.append({
            "year": year,
            f"{prefix}_ann": ann if pd.notna(ann) else 0.0,
            f"{prefix}_jja": jja if pd.notna(jja) else 0.0,
            f"{prefix}_son": son if pd.notna(son) else 0.0,
        })
    return pd.DataFrame(rows)


nino_feat = annual_features(nino, "nino34")
dmi_feat  = annual_features(dmi,  "dmi")

# Historical climate covariates: only up through 2025.
climate_obs = (
    nino_feat.merge(dmi_feat, on="year", how="outer")
    .query("year >= 2010 and year <= 2025")
    .reset_index(drop=True)
)
print("\nHistorical annual climate features (last 10 years):")
print(climate_obs.tail(10).to_string(index=False))

# Scenario-conditional climate values for 2026..2030 — these are ASSUMPTIONS,
# never fetched.
climate_scenarios = {
    name: pd.DataFrame({
        "year": FORECAST_YEARS,
        "nino34_ann": s["nino34_jja"],
        "nino34_jja": s["nino34_jja"],
        "nino34_son": s["nino34_jja"],
        "dmi_ann":    s["dmi_jja_offset"],
        "dmi_jja":    s["dmi_jja_offset"],
        "dmi_son":    s["dmi_jja_offset"],
    })
    for name, s in SCENARIOS.items()
}
print(f"\nScenario climate assumptions defined for {FORECAST_YEARS[0]}..{FORECAST_YEARS[-1]}: "
      f"{list(climate_scenarios.keys())}")

In [ ]:
ELEV_CACHE = CACHE_DIR / "elevation.parquet"

def fetch_elevations(lats: list[float], lngs: list[float]) -> list[float]:
    elevs: list[float] = []
    # Open-Meteo accepts up to 100 lat/lng pairs per call.
    for start in range(0, len(lats), 100):
        chunk_lat = lats[start:start+100]
        chunk_lng = lngs[start:start+100]
        url = (
            "https://api.open-meteo.com/v1/elevation"
            f"?latitude={','.join(f'{x:.6f}' for x in chunk_lat)}"
            f"&longitude={','.join(f'{x:.6f}' for x in chunk_lng)}"
        )
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        elevs.extend(resp.json()["elevation"])
    return elevs


if ELEV_CACHE.exists():
    elev_df = pd.read_parquet(ELEV_CACHE)
    print(f"Elevation: loaded from cache ({len(elev_df)} towers)")
else:
    try:
        elevs = fetch_elevations(towers["lat"].tolist(), towers["lng"].tolist())
        elev_df = towers[["tower_id"]].copy()
        elev_df["elev_m"] = elevs
        elev_df.to_parquet(ELEV_CACHE)
        print(f"Elevation: fetched fresh for {len(elev_df)} towers")
    except Exception as exc:
        print(f"Elevation fetch failed ({exc}); using lat-linear placeholder.")
        elev_df = towers[["tower_id"]].copy()
        elev_df["elev_m"] = 50.0  # benign constant placeholder


def haversine_km(lat1: float, lng1: float, lat2: float, lng2: float) -> float:
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lng2 - lng1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlmb/2)**2
    return 2 * R * math.asin(math.sqrt(a))


# Anchor points: Bali Strait east edge and Java Sea north edge.
BALI_STRAIT = (-8.15, 114.45)
JAVA_SEA    = (-7.70, 114.20)

def coast_distances(row: pd.Series) -> pd.Series:
    return pd.Series({
        "dist_strait_km": haversine_km(row["lat"], row["lng"], *BALI_STRAIT),
        "dist_jsea_km":   haversine_km(row["lat"], row["lng"], *JAVA_SEA),
    })

spatial = towers.merge(elev_df, on="tower_id", how="left")
spatial = pd.concat([spatial, spatial.apply(coast_distances, axis=1)], axis=1)
spatial["dist_coast_km"] = spatial[["dist_strait_km", "dist_jsea_km"]].min(axis=1)

print("\nSpatial covariates summary:")
print(spatial[["elev_m", "dist_strait_km", "dist_jsea_km", "dist_coast_km"]].describe().round(2).to_string())

In [ ]:
panel_m = panel.merge(spatial.drop(columns=["asset_name"]), on="tower_id", how="left")
panel_m = panel_m.merge(climate_obs, on="year", how="left")

lat_mean = float(spatial["lat"].mean())
lng_mean = float(spatial["lng"].mean())

panel_m["lat_c"]      = panel_m["lat"] - lat_mean
panel_m["lng_c"]      = panel_m["lng"] - lng_mean
panel_m["log_elev"]   = np.log(panel_m["elev_m"].clip(lower=1.0))
panel_m["enso_x_lat"] = panel_m["nino34_jja"] * panel_m["lat_c"]
panel_m["year_idx"]   = panel_m["year"] - HIST_YEARS[0]

print(f"X_train: {panel_m.shape[0]} rows, {panel_m.shape[1]} cols")

# Build X_future once per scenario.
X_future_by_scenario: dict[str, pd.DataFrame] = {}
for sc_name, sc_clim in climate_scenarios.items():
    grid = (
        spatial.merge(pd.DataFrame({"year": FORECAST_YEARS}), how="cross")
        .merge(sc_clim, on="year", how="left")
    )
    grid["lat_c"]      = grid["lat"] - lat_mean
    grid["lng_c"]      = grid["lng"] - lng_mean
    grid["log_elev"]   = np.log(grid["elev_m"].clip(lower=1.0))
    grid["enso_x_lat"] = grid["nino34_jja"] * grid["lat_c"]
    grid["year_idx"]   = grid["year"] - HIST_YEARS[0]
    X_future_by_scenario[sc_name] = grid
    print(f"  X_future[{sc_name}]: {grid.shape}")

## Section C — Exploratory diagnostics

We confirm the key empirical patterns that the modeling choices depend on:

1. **Year-to-year line totals correlate with ENSO/IOD** → supports climate
   covariates.
2. **Per-tower-year Count is overdispersed** → reject Poisson, use NB.
3. **Spatial pattern is stable across years** → supports Model C's
   per-tower-share allocation.
4. **Hotspots cluster geographically** → supports the spatial smoother.

In [ ]:
# (1) Annual line totals with ENSO overlay
yearly = panel_m.groupby("year").agg(
    line_count=("count", "sum"),
    line_density=("density", "mean"),
    nino34_jja=("nino34_jja", "first"),
    dmi_jja=("dmi_jja", "first"),
).reset_index()

corr_n = yearly[["line_count", "nino34_jja"]].corr().iloc[0, 1]
corr_d = yearly[["line_count", "dmi_jja"]].corr().iloc[0, 1]
print(f"Line-count vs Niño 3.4 (JJA) Pearson r = {corr_n:+.3f}")
print(f"Line-count vs IOD DMI (JJA) Pearson r = {corr_d:+.3f}")
print()

fig = go.Figure()
fig.add_trace(go.Scatter(x=yearly["year"], y=yearly["line_count"],
                         name="Line strikes", mode="lines+markers", yaxis="y"))
fig.add_trace(go.Scatter(x=yearly["year"], y=yearly["nino34_jja"],
                         name="Niño 3.4 (JJA)", mode="lines+markers", yaxis="y2"))
fig.add_trace(go.Scatter(x=yearly["year"], y=yearly["dmi_jja"],
                         name="IOD DMI (JJA)", mode="lines+markers", yaxis="y2"))
fig.update_layout(
    title="Annual line strikes vs ENSO / IOD",
    xaxis_title="Year",
    yaxis=dict(title="Strikes (count)"),
    yaxis2=dict(title="Climate index (°C)", overlaying="y", side="right"),
    height=400, legend=dict(orientation="h", y=-0.2),
)
fig.show()

# (2) Overdispersion check — variance/mean ratio per year
disp = panel_m.groupby("year")["count"].agg(["mean", "var"]).assign(
    ratio=lambda d: d["var"] / d["mean"]
)
print("Per-tower-year Count, by year — variance/mean ratio (>>1 ⇒ overdispersed):")
print(disp.round(2).to_string())

# Fit Poisson vs NB to all rows; report log-likelihoods.
y = panel_m["count"].values
poisson_ll = sm.GLM(y, np.ones_like(y), family=sm.families.Poisson()).fit().llf
nb_alpha = max(0.01, (panel_m["count"].var() - panel_m["count"].mean()) / panel_m["count"].mean()**2)
nb_ll = sm.GLM(y, np.ones_like(y), family=sm.families.NegativeBinomial(alpha=nb_alpha)).fit().llf
print(f"\nIntercept-only LL — Poisson: {poisson_ll:.0f}, NB(alpha={nb_alpha:.3f}): {nb_ll:.0f}")
print(f"NB beats Poisson by {nb_ll - poisson_ll:.0f} log-likelihood units → confirm NB is appropriate.")

# (3) Per-tower share stability across years (drives Model C)
share = panel_m.assign(line_total=lambda d: d.groupby("year")["count"].transform("sum"))
share["share"] = share["count"] / share["line_total"]
share_cv = share.groupby("tower_id")["share"].agg(["mean", "std"]).assign(
    cv=lambda d: d["std"] / d["mean"].replace(0, np.nan)
)
print(f"\nPer-tower share-of-line CV across 7 years:")
print(f"  median CV = {share_cv['cv'].median():.3f}")
print(f"  fraction with CV < 0.50 = {(share_cv['cv'] < 0.50).mean():.2%}")
print(f"  fraction with CV < 0.30 = {(share_cv['cv'] < 0.30).mean():.2%}")

## Section D — Candidate models

We fit two models:

- **Model A — Climatology** (per-tower 7-year mean). The skill floor.
- **Model C — Two-stage hybrid (PRIMARY)**. Stage 1: line-level NB regression
  of annual total on lagged ENSO/IOD (AICc-selected, ≤3 predictors at n=7).
  Stage 2: empirical-Bayes shrunken per-tower share-of-line.

**Model B (NB GLM with spatial smoother) was attempted and dropped** —
its expanding-window CV-RMSE was 3,139 (vs ~14 for A and C), a perfect-
separation catastrophe driven by the spline basis overwhelming the
training-fold size. With only 7 yearly observations, even an additive
spline on (lat, lng) plus climate covariates is too parameter-heavy to
fit reliably. The Model B code path is removed from the deliverables.

### Units — read this before interpreting any output

The dataset has two different "totals" that must not be conflated:

| Quantity | Units | Where it comes from | What it represents |
|---|---|---|---|
| `density` (per tower) | flashes / km² / yr | per-tower row in Exposure | overlap-counted per-tower strike rate = `count_per_tower / 3.125 km²`. ~5× the IEEE 1410 single-flash GFD because each strike falls within ~5 adjacent towers' 3.125 km² circles. Useful for **relative per-tower ranking and per-tower induced-voltage risk** (every strike near a tower stresses it regardless of how neighbors counted it). |
| `density` (aggregate row) | flashes / km² / yr | "All selected assets" row in Exposure | **IEEE 1410 single-flash GFD** = unique strikes ÷ total line area. Comparable to published climatologies. |
| `count` (per tower) | strikes / yr | per-tower row in Exposure | overlapping — same caveat as above |
| `count` (aggregate row) | strikes / yr | "All selected assets" row | unique strikes near the line (~1/5 of panel sum) |
| `count` (sum of panel) | strikes / yr | `panel.groupby('year').count.sum()` | sum of overlapping per-tower counts. **Not** a count of distinct lightning events. |

**Reconciliation** (verified in the executed notebook): per-tower density mean ÷ aggregate density = 4.93–5.19 across 2019–2025. The factor of ~5 is the average number of tower collection-circles each strike falls into.

For each tower-year we forecast three targets:

1. **`density`** (GFD, flashes/km²/yr) — the headline IEEE 1410 quantity, the
   user's stated primary target.
2. `count` per tower (overlapping) — kept for traceability to source data.
3. `mean_ka` / `max_ka` — line-level kA shift × per-tower offset.

In [ ]:
def model_a_climatology(train: pd.DataFrame, future_years: Sequence[int]) -> pd.DataFrame:
    """Per-tower 7-year mean of each target, repeated across future years."""
    per_tower = train.groupby("tower_id")[["count","density","mean_ka","max_ka"]].mean()
    return per_tower.assign(key=1).reset_index().merge(
        pd.DataFrame({"year": list(future_years), "key": 1}), on="key"
    ).drop(columns="key")


def model_a_enso_analog(train: pd.DataFrame, future_climate: pd.DataFrame,
                         k: int = 3) -> pd.DataFrame:
    """ENSO-analog-year variant: per forecast year, average over the k
    historical years with the nearest Niño 3.4 JJA value."""
    hist_clim = train.groupby("year")["nino34_jja"].first()
    out = []
    for _, fr in future_climate.iterrows():
        target = fr["nino34_jja"]
        analog_years = (hist_clim - target).abs().sort_values().index[:k].tolist()
        means = train[train["year"].isin(analog_years)].groupby("tower_id")[
            ["count","density","mean_ka","max_ka"]
        ].mean()
        out.append(means.assign(year=int(fr["year"])).reset_index())
    return pd.concat(out, ignore_index=True)


forecast_A_clim = model_a_climatology(panel_m, FORECAST_YEARS)
print(f"Model A (climatology): {forecast_A_clim.shape} (281 × 5 = 1,405 rows)")
print(forecast_A_clim.groupby("year")["count"].sum().round(0).to_string())

In [ ]:
# Model B was originally an NB GLM with cubic-regression spline smoother on
# (lat, lng) plus climate and topographic covariates. In CV it produced
# expanding-window RMSE of ~3,139 — a perfect-separation catastrophe — vs
# ~14 for Models A and C. With only 7 yearly observations, even an additive
# spline basis is too parameter-heavy. Model B is intentionally removed
# from the deliverables.
print("Model B (NB GLM + spatial smoother) — intentionally not fit.")
print("  Reason: expanding-window CV failed (RMSE ~3,139 vs ~14 for A and C)")
print("  due to perfect separation at n=7 yearly observations.")
print("  See plan file / notebook intro for the dropped-model rationale.")

In [ ]:
@dataclass
class ModelCFit:
    line_count_model: sm.GLM
    line_count_alpha: float
    line_kamean_model: sm.GLM
    line_kamax_model:  sm.GLM
    shares_count:   pd.Series   # tower_id -> mean share (count)
    shares_density: pd.Series   # tower_id -> mean share (density)
    shares_kamean:  pd.Series   # tower_id -> per-tower mean kA centered on line
    shares_kamax:   pd.Series


def _james_stein_shrinkage(shares: pd.DataFrame, prior_mean: pd.Series | None = None) -> pd.Series:
    """Empirical-Bayes shrinkage of per-tower per-year share toward an
    overall prior. Operates on the long share table with columns
    [tower_id, year, share].
    """
    grand = shares["share"].mean()
    target = prior_mean if prior_mean is not None else pd.Series(grand, index=shares["tower_id"].unique())
    tower_mean = shares.groupby("tower_id")["share"].mean()
    tower_var  = shares.groupby("tower_id")["share"].var(ddof=1).fillna(0)
    # Stein-style weight: w_i = sigma2_w / (sigma2_w + sigma2_b)
    between_var = tower_mean.var(ddof=1)
    w = between_var / (between_var + tower_var / max(shares["year"].nunique(), 2))
    shrunk = w * tower_mean + (1 - w) * target.reindex(tower_mean.index).fillna(grand)
    shrunk = shrunk / shrunk.sum()
    return shrunk


def fit_model_c(train_df: pd.DataFrame, line_df: pd.DataFrame, climate_df: pd.DataFrame) -> ModelCFit:
    # ---------- Stage 1: line-level temporal models ----------
    # IMPORTANT: the "Exposure" aggregate row counts UNIQUE strikes near the
    # line, while the sum of per-tower counts is the OVERLAPPING total
    # (each strike falls within ~5 adjacent towers' 3.125 km^2 circles).
    # We allocate back to per-tower counts, so Stage 1 must fit the OVERLAPPING
    # total — derived from the panel — not the aggregate row.
    line_overlap = (
        train_df.groupby("year", as_index=False)
                .agg(count=("count", "sum"),
                     mean_ka=("mean_ka", "mean"),
                     max_ka=("max_ka", "mean"))
    )
    line = line_overlap.merge(climate_df, on="year", how="left").sort_values("year").reset_index(drop=True)
    # Pick best of {intercept-only, +nino, +nino+dmi, +nino+year_idx} by AICc
    line["year_idx"] = line["year"] - HIST_YEARS[0]
    candidates = [
        "count ~ 1",
        "count ~ nino34_jja",
        "count ~ nino34_jja + dmi_jja",
        "count ~ nino34_jja + year_idx",
    ]
    pois_alpha = max((line["count"].var() - line["count"].mean()) / line["count"].mean()**2, 0.05)
    best, best_aic = None, np.inf
    for formula in candidates:
        try:
            m = smf.glm(formula, data=line, family=sm.families.NegativeBinomial(alpha=pois_alpha)).fit()
            k = m.df_model + 1
            n = len(line)
            aicc = m.aic + 2*k*(k+1) / max(n - k - 1, 1)
            if aicc < best_aic:
                best, best_aic = (formula, m), aicc
        except Exception:
            continue
    formula, line_count_model = best
    print(f"Stage 1 (line Count): selected formula '{formula}'  AICc={best_aic:.1f}")
    print(f"  coefficients: {dict(zip(line_count_model.params.index, line_count_model.params.round(3)))}")

    # Line-level kA: simple regression on Niño 3.4 — magnitudes shift in El Niño years.
    line_kamean_model = smf.ols("mean_ka ~ nino34_jja", data=line).fit()
    line_kamax_model  = smf.ols("max_ka ~ nino34_jja",  data=line).fit()
    print(f"Stage 1 (line mean_ka): coef nino34_jja = {line_kamean_model.params['nino34_jja']:+.2f} kA")
    print(f"Stage 1 (line max_ka):  coef nino34_jja = {line_kamax_model.params['nino34_jja']:+.2f} kA")

    # ---------- Stage 2: per-tower shares ----------
    shares_long = train_df.assign(
        share_count=lambda d: d["count"] / d.groupby("year")["count"].transform("sum"),
    )[["tower_id", "year", "share_count"]].rename(columns={"share_count": "share"})
    shares_count = _james_stein_shrinkage(shares_long)

    shares_long_d = train_df.assign(
        share_density=lambda d: d["density"] / d.groupby("year")["density"].transform("sum"),
    )[["tower_id", "year", "share_density"]].rename(columns={"share_density": "share"})
    shares_density = _james_stein_shrinkage(shares_long_d)

    # Per-tower kA centered on line mean (residual kA).
    tower_kamean = train_df.groupby("tower_id")["mean_ka"].mean()
    line_kamean  = train_df.groupby("year")["mean_ka"].mean().mean()
    shares_kamean = tower_kamean - line_kamean
    tower_kamax  = train_df.groupby("tower_id")["max_ka"].mean()
    line_kamax   = train_df.groupby("year")["max_ka"].mean().mean()
    shares_kamax = tower_kamax - line_kamax

    return ModelCFit(line_count_model, pois_alpha,
                     line_kamean_model, line_kamax_model,
                     shares_count, shares_density,
                     shares_kamean, shares_kamax)


def predict_model_c(fit: ModelCFit, X: pd.DataFrame) -> pd.DataFrame:
    # X has tower-year-scenario rows; line-level predictors are constant within (year,scenario).
    line_X = X[["year", "nino34_jja", "dmi_jja"]].drop_duplicates()
    line_X["year_idx"] = line_X["year"] - HIST_YEARS[0]
    line_X["count_hat"]   = fit.line_count_model.predict(line_X)
    line_X["mean_ka_hat"] = fit.line_kamean_model.predict(line_X)
    line_X["max_ka_hat"]  = fit.line_kamax_model.predict(line_X)

    out = X[["tower_id", "year"]].merge(line_X, on="year", how="left")
    out["count"]   = out["count_hat"]   * out["tower_id"].map(fit.shares_count).fillna(1/281)
    out["density"] = out["count"] / COLLECTION_AREA_KM2
    out["mean_ka"] = out["mean_ka_hat"] + out["tower_id"].map(fit.shares_kamean).fillna(0)
    out["max_ka"]  = out["max_ka_hat"]  + out["tower_id"].map(fit.shares_kamax).fillna(0)
    return out[["tower_id", "year", "count", "density", "mean_ka", "max_ka"]]


fit_C = fit_model_c(panel_m, line_year, climate_obs)
forecast_C = pd.concat(
    [predict_model_c(fit_C, df).assign(scenario=name)
     for name, df in X_future_by_scenario.items()],
    ignore_index=True,
)
print(f"\nModel C forecast: {forecast_C.shape}")
print("Model C line totals by scenario × year:")
print(forecast_C.groupby(["scenario","year"])["count"].sum().round(0).unstack(0).to_string())

## Section E — Cross-validation and skill

Two CV schemes:

- **Leave-One-Year-Out (LOYO)** — 7 folds, each year held out once. The natural
  scheme for our 7-year dataset; tells us how skillful the model would have
  been if a held-out year arrived next.
- **Expanding-window walk-forward** — 6 folds: train {2019}, predict 2020;
  train {2019,2020}, predict 2021; …; train {2019..2024}, predict 2025.
  Mimics operational deployment.

Metrics: **MAE**, **RMSE**, **Poisson deviance** (Count), and **CRPS**
(probabilistic). All reported per (target × model × CV-scheme), plus a
**skill score** against Model A (climatology).

In [ ]:
def metric_block(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float).clip(min=1e-6)
    mae  = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred)**2)))
    with np.errstate(divide="ignore", invalid="ignore"):
        dev_term = np.where(y_true > 0, y_true * np.log(y_true / y_pred), 0.0) - (y_true - y_pred)
    deviance = float(2 * np.sum(dev_term))
    # Empirical Gaussian-CRPS (closed form) with sigma = max(0.5, sqrt(y_pred)).
    sigma = np.maximum(0.5, np.sqrt(y_pred))
    z = (y_true - y_pred) / sigma
    crps = float(np.mean(sigma * (z*(2*st.norm.cdf(z) - 1) + 2*st.norm.pdf(z) - 1/np.sqrt(np.pi))))
    return {"MAE": mae, "RMSE": rmse, "Deviance": deviance, "CRPS": crps}


def cv_loyo(train_df: pd.DataFrame, line_df: pd.DataFrame, climate_df: pd.DataFrame,
            target: str = "count") -> pd.DataFrame:
    rows = []
    for held in HIST_YEARS:
        tr = train_df[train_df["year"] != held].copy()
        te = train_df[train_df["year"] == held].copy()
        tr_line = line_df[line_df["year"] != held]

        # Model A (climatology of training years)
        means = tr.groupby("tower_id")[target].mean()
        pA = te["tower_id"].map(means).fillna(means.mean()).values

        # Model C
        try:
            fC = fit_model_c(tr, tr_line, climate_df)
            pC = predict_model_c(fC, te.assign(year=held).merge(
                climate_df, on="year", how="left", suffixes=("","_dup")))[target].values
        except Exception:
            pC = np.full(len(te), np.nan)

        y = te[target].values
        for name, pred in [("A", pA), ("C", pC)]:
            if np.all(np.isfinite(pred)):
                m = metric_block(y, pred)
                m.update({"model": name, "year_held": held, "target": target, "scheme": "LOYO"})
                rows.append(m)
    return pd.DataFrame(rows)


def cv_expanding(train_df: pd.DataFrame, line_df: pd.DataFrame, climate_df: pd.DataFrame,
                 target: str = "count") -> pd.DataFrame:
    rows = []
    for cutoff in HIST_YEARS[:-1]:
        tr = train_df[train_df["year"] <= cutoff].copy()
        te_year = cutoff + 1
        if te_year not in HIST_YEARS:
            continue
        te = train_df[train_df["year"] == te_year].copy()
        tr_line = line_df[line_df["year"] <= cutoff]

        means = tr.groupby("tower_id")[target].mean()
        pA = te["tower_id"].map(means).fillna(means.mean()).values

        try:
            fC = fit_model_c(tr, tr_line, climate_df)
            pC = predict_model_c(fC, te.merge(climate_df, on="year", how="left", suffixes=("","_dup")))[target].values
        except Exception:
            pC = np.full(len(te), np.nan)

        y = te[target].values
        for name, pred in [("A", pA), ("C", pC)]:
            if np.all(np.isfinite(pred)):
                m = metric_block(y, pred)
                m.update({"model": name, "year_held": te_year, "target": target, "scheme": "Expanding"})
                rows.append(m)
    return pd.DataFrame(rows)


cv_count_loyo = cv_loyo(panel_m, line_year, climate_obs, "count")
cv_count_exp  = cv_expanding(panel_m, line_year, climate_obs, "count")
cv_dens_loyo  = cv_loyo(panel_m, line_year, climate_obs, "density")
cv_kamean_loyo = cv_loyo(panel_m, line_year, climate_obs, "mean_ka")

cv_all = pd.concat([cv_count_loyo, cv_count_exp, cv_dens_loyo, cv_kamean_loyo], ignore_index=True)
cv_summary = (cv_all.groupby(["scheme", "target", "model"])
                    [["MAE", "RMSE", "Deviance", "CRPS"]]
                    .mean().round(3))

# Skill score vs Model A.
def skill_vs_a(df: pd.DataFrame, metric: str) -> pd.Series:
    a = df.xs("A", level="model")[metric]
    return 1 - df[metric].unstack("model").div(a, axis=0)

skill = pd.concat({
    "skill_RMSE": skill_vs_a(cv_summary, "RMSE"),
    "skill_CRPS": skill_vs_a(cv_summary, "CRPS"),
}, axis=1).round(3)

print("=" * 70)
print("CV summary (mean of folds):")
print(cv_summary.to_string())
print()
print("Skill score vs Model A (positive = better than climatology):")
print(skill.to_string())
cv_all.to_parquet(OUT_DIR / "cv_results.parquet")

# 2025 cross-check: train on 2019..2024, predict 2025.
tr24 = panel_m[panel_m["year"] <= 2024]
te25 = panel_m[panel_m["year"] == 2025]
fit_C25 = fit_model_c(tr24, line_year[line_year["year"] <= 2024], climate_obs)
pred25 = predict_model_c(fit_C25, te25.merge(climate_obs, on="year", how="left", suffixes=("","_dup")))
merged = te25[["tower_id","count"]].merge(pred25[["tower_id","count"]], on="tower_id", suffixes=("_obs","_pred"))
r = merged[["count_obs","count_pred"]].corr().iloc[0,1]
mae25 = (merged["count_obs"] - merged["count_pred"]).abs().mean()
print(f"\n2025 hold-out (Model C trained on 2019..2024):  Pearson r = {r:+.3f},  MAE = {mae25:.2f}")

In [ ]:
def model_c_bootstrap_one(seed: int, train_df: pd.DataFrame, line_df: pd.DataFrame,
                          climate_df: pd.DataFrame,
                          X_future: pd.DataFrame) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    # Resample years with replacement (block bootstrap on the year dimension).
    sampled_years = rng.choice(HIST_YEARS, size=len(HIST_YEARS), replace=True)
    tr = pd.concat([train_df[train_df["year"] == y] for y in sampled_years], ignore_index=True)
    tr_line = pd.concat([line_df[line_df["year"] == y] for y in sampled_years], ignore_index=True)
    try:
        fit = fit_model_c(tr, tr_line, climate_df)
        pred = predict_model_c(fit, X_future)
        # Negative-binomial process noise on top of the mean.
        alpha = fit.line_count_alpha
        for col in ["count"]:
            mu = pred[col].clip(lower=1e-3)
            # NB variance = mu + alpha * mu^2; sample from gamma-poisson mix.
            shape = 1 / alpha
            scale = alpha * mu
            lam = rng.gamma(shape=shape, scale=scale)
            pred[col] = rng.poisson(lam).astype(float)
            pred["density"] = pred["count"] / COLLECTION_AREA_KM2
        # Gaussian noise for kA columns.
        for col in ["mean_ka", "max_ka"]:
            pred[col] = pred[col] + rng.normal(0, 3.0, size=len(pred))
        return pred
    except Exception:
        return pd.DataFrame()


print(f"Running {N_BOOTSTRAP}-rep bootstrap × {len(SCENARIOS)} scenarios...")

all_samples = []
for sc_name, X_fut in X_future_by_scenario.items():
    samples = Parallel(n_jobs=-1, verbose=0)(
        delayed(model_c_bootstrap_one)(
            RNG_SEED + sc_idx*10000 + i,
            panel_m, line_year, climate_obs, X_fut,
        )
        for i, sc_idx in zip(range(N_BOOTSTRAP), [list(SCENARIOS).index(sc_name)] * N_BOOTSTRAP)
    )
    samples = [s.assign(rep=i, scenario=sc_name) for i, s in enumerate(samples) if not s.empty]
    print(f"  {sc_name}: {len(samples)}/{N_BOOTSTRAP} successful bootstrap reps")
    all_samples.extend(samples)

boot_df = pd.concat(all_samples, ignore_index=True)
print(f"Total bootstrap rows: {len(boot_df):,}")


def summarize_interval(g: pd.DataFrame, col: str) -> pd.Series:
    return pd.Series({
        f"{col}_p50":  np.median(g[col]),
        f"{col}_lo80": np.quantile(g[col], 0.10),
        f"{col}_hi80": np.quantile(g[col], 0.90),
        f"{col}_lo95": np.quantile(g[col], 0.025),
        f"{col}_hi95": np.quantile(g[col], 0.975),
    })


summaries = []
for col in ["count", "density", "mean_ka", "max_ka"]:
    s = (boot_df.groupby(["tower_id", "year", "scenario"])
                .apply(summarize_interval, col=col, include_groups=False)
                .reset_index())
    summaries.append(s)

forecast_intervals = summaries[0]
for s in summaries[1:]:
    forecast_intervals = forecast_intervals.merge(s, on=["tower_id", "year", "scenario"])

# Climate-marginalized interval = pool scenarios with equal weight.
climate_marg = (boot_df.groupby(["tower_id", "year"])
                       .apply(lambda g: pd.concat([summarize_interval(g, c)
                                                   for c in ["count","density","mean_ka","max_ka"]]),
                              include_groups=False)
                       .reset_index())
climate_marg["scenario"] = "Marginalized"
forecast_intervals = pd.concat(
    [forecast_intervals, climate_marg[forecast_intervals.columns]],
    ignore_index=True,
)
forecast_intervals.to_parquet(OUT_DIR / "forecast_2026_2030.parquet")
print(f"\nForecast intervals: {forecast_intervals.shape}")
print("Sample (tower 1):")
print(forecast_intervals.query("tower_id == 1 and scenario == 'Neutral'")
      [["year","count_p50","count_lo80","count_hi80","density_p50"]]
      .round(2).to_string(index=False))

In [ ]:
# ---------- 1. Multi-sheet Excel ----------
xlsx_path = OUT_DIR / "per_tower_forecast.xlsx"

# Build dedicated GFD sheets (the primary deliverable) with unambiguous
# column names. Each row is (tower_id, year) for one scenario.
gfd_cols = ["density_p50", "density_lo80", "density_hi80", "density_lo95", "density_hi95"]
count_cols = ["count_p50", "count_lo80", "count_hi80", "count_lo95", "count_hi95"]
ka_cols = ["mean_ka_p50", "mean_ka_lo80", "mean_ka_hi80",
           "max_ka_p50",  "max_ka_lo80",  "max_ka_hi80"]

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xw:
    readme = pd.DataFrame({
        "field": [
            "PRIMARY DELIVERABLE",
            "GFD_<scenario>.density_p50",
            "GFD_<scenario>.density_lo80 / hi80",
            "",
            "SECONDARY (kept for traceability)",
            "Count_<scenario>.count_p50",
            "kA_<scenario>.mean_ka_p50, max_ka_p50",
            "",
            "GLOBAL",
            "tower_id",
            "year",
            "scenario",
            "",
            "MODELS",
            "Primary model",
            "Model A baseline",
            "Model B",
            "",
            "CAVEATS (read before using)",
            "n_years",
            "Skill vs climatology",
            "Stage-1 selection",
            "2025 hold-out",
            "Stationarity",
            "Unique-strike line total",
            "",
        ],
        "meaning": [
            "",
            "Per-tower density forecast, flashes / km² / yr. NOTE: this is "
            "the OVERLAP-COUNTED rate from the source data (each unique "
            "strike falls within ~5 adjacent towers' 3.125 km² collection "
            "circles, so this value is ~5× the IEEE 1410 single-flash GFD "
            "for the same area). Use this for: relative per-tower risk "
            "ranking, per-tower induced-voltage stress assessment, surge-"
            "arrester sizing at the tower. Do NOT use directly for "
            "comparison to published GFD climatologies — divide by ~5 first.",
            "80% prediction interval (10th / 90th percentile of bootstrap).",
            "",
            "",
            "Per-tower overlapping strike count, strikes / yr. Each "
            "physical lightning event is counted by every tower within its "
            "3.125 km² collection area (~5 adjacent towers per strike). "
            "**Not** a count of distinct flashes; do not sum across towers "
            "to get total events on the line.",
            "Mean / max peak current at the tower, kA. Sign convention: "
            "negative = negative polarity (downward strokes).",
            "",
            "",
            "Tower id 1..281, matches `data_tidy/towers.csv`.",
            "Forecast year 2026..2030.",
            "Climate scenario: LaNina / Neutral / ElNino, or "
            "'Marginalized' (equal-weight pool of the three).",
            "",
            "",
            "Model C — two-stage hybrid: NB regression on line-level "
            "annual total + empirical-Bayes shrunken per-tower share.",
            "Per-tower 7-year mean (climatology). Reported alongside as "
            "the skill floor.",
            "INTENTIONALLY DROPPED. Original NB GLM with spatial smoother "
            "failed CV (RMSE ~3,139 vs ~14). Not in any deliverable sheet.",
            "",
            "",
            "n=7 yearly observations per tower. Forecast horizon = 5 years. "
            "Very small training sample relative to noise.",
            f"Model C beats climatology by ~2% RMSE in CV. Marginal — "
            f"treat as a smarter climatology, not a skillful climate forecast.",
            "Stage-1 AICc selection favored intercept-only (no climate "
            "covariates) on the full 7-year fit. Climate signal could not "
            "statistically earn its place at this sample size.",
            "2025 held out: Pearson r between predicted and observed "
            "per-tower count = +0.237. Below the 0.6 threshold set in "
            "the plan. Per-tower allocation is noisier than hoped.",
            "Intervals assume GFD-climate relationship is stationary over "
            "7 years. Decadal-scale trends are NOT captured.",
            "If you need unique-flash totals near the line (for LDN "
            "comparison or pole-failure rate estimation), use the "
            "aggregate-row column in `data_tidy/exposure_line_year.csv` "
            "(roughly 1/5 of the sum-of-per-tower-counts).",
            "",
        ],
    })
    readme.to_excel(xw, sheet_name="README", index=False)

    for sc in ["LaNina", "Neutral", "ElNino", "Marginalized"]:
        sl = forecast_intervals.query("scenario == @sc").copy()
        sl[["tower_id","year"] + gfd_cols].to_excel(
            xw, sheet_name=f"GFD_{sc}", index=False
        )
        sl[["tower_id","year"] + count_cols].to_excel(
            xw, sheet_name=f"Count_{sc}", index=False
        )
        sl[["tower_id","year"] + ka_cols].to_excel(
            xw, sheet_name=f"kA_{sc}", index=False
        )
print(f"Wrote {xlsx_path}  ({xlsx_path.stat().st_size:,} bytes)")

# ---------- 2. Folium GFD hotspot map ----------
center_lat = float(spatial["lat"].mean())
center_lng = float(spatial["lng"].mean())
m = folium.Map(location=[center_lat, center_lng], zoom_start=11, tiles="OpenStreetMap")

line_coords = spatial.sort_values("tower_id")[["lat","lng"]].values.tolist()
folium.PolyLine(line_coords, color="#444", weight=2, opacity=0.6,
                tooltip="150 kV Situbondo–Banyuwangi line").add_to(m)

# Per-scenario layer: 5-year MEAN GFD (flashes/km²/yr averaged over 2026..2030).
gfd_5yr = (forecast_intervals
           .query("scenario != 'Marginalized'")
           .groupby(["tower_id","scenario"])
           .agg(gfd_mean=("density_p50","mean"),
                gfd_lo=("density_lo80","mean"),
                gfd_hi=("density_hi80","mean"))
           .reset_index())

size_max = gfd_5yr["gfd_mean"].max()
size_min = gfd_5yr["gfd_mean"].min()

for sc in ["LaNina", "Neutral", "ElNino"]:
    fg = folium.FeatureGroup(name=f"5-yr mean GFD ({sc})")
    cluster = MarkerCluster(disableClusteringAtZoom=12).add_to(fg)
    layer = gfd_5yr.query("scenario == @sc").merge(
        spatial[["tower_id","lat","lng","elev_m"]], on="tower_id")
    for _, r in layer.iterrows():
        frac = (r["gfd_mean"] - size_min) / max(size_max - size_min, 1)
        radius = 3 + 9 * frac
        color = f"#{int(255*frac):02x}00{int(255*(1-frac)):02x}"
        folium.CircleMarker(
            location=[r["lat"], r["lng"]],
            radius=radius,
            color=color, fill=True, fill_color=color, fill_opacity=0.75,
            popup=(f"<b>Tower {int(r['tower_id'])}</b><br>"
                   f"Scenario: {sc}<br>"
                   f"5-yr mean <b>GFD = {r['gfd_mean']:.2f}</b> flashes/km²/yr<br>"
                   f"80% PI: [{r['gfd_lo']:.2f}, {r['gfd_hi']:.2f}]<br>"
                   f"Elev: {r['elev_m']:.0f} m"),
        ).add_to(cluster)
    fg.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

map_path = OUT_DIR / "hotspot_map.html"
m.save(str(map_path))
print(f"Wrote {map_path}  ({map_path.stat().st_size:,} bytes)")

# ---------- 3. Top-20 ranking by 5-year mean GFD ----------
ranking = (forecast_intervals.query("scenario == 'Neutral'")
           .groupby("tower_id")
           .agg(gfd_mean=("density_p50","mean"),
                gfd_lo80=("density_lo80","mean"),
                gfd_hi80=("density_hi80","mean"))
           .reset_index()
           .merge(spatial[["tower_id","lat","lng","elev_m"]], on="tower_id")
           .sort_values("gfd_mean", ascending=False)
           .head(20))
ranking["units"] = "flashes/km^2/yr (5-yr mean, Neutral scenario)"
ranking.to_csv(OUT_DIR / "top20_ranking.csv", index=False)
print(f"Wrote {OUT_DIR / 'top20_ranking.csv'}")

fig_rank = go.Figure(go.Bar(
    x=ranking["gfd_mean"],
    y=ranking["tower_id"].astype(str),
    orientation="h",
    error_x=dict(type="data",
                 array=ranking["gfd_hi80"] - ranking["gfd_mean"],
                 arrayminus=ranking["gfd_mean"] - ranking["gfd_lo80"]),
    text=ranking["gfd_mean"].round(2),
    textposition="outside",
))
fig_rank.update_layout(
    title="Top 20 towers by 5-year mean GFD (Neutral scenario, 80% PI)",
    xaxis_title="GFD (flashes/km²/yr, 5-year mean)",
    yaxis_title="Tower ID",
    yaxis=dict(autorange="reversed"),
    height=600,
)
fig_rank.show()

# ---------- 4. CV skill HTML ----------
html_path = OUT_DIR / "cv_skill.html"
(cv_summary
    .style.background_gradient(subset=["RMSE","CRPS"], cmap="RdYlGn_r")
    .set_caption("Cross-validation skill (mean of folds)")
    .to_html(html_path))
print(f"Wrote {html_path}")

# ---------- 5. Sensitivity table (per-tower GFD response to ENSO) ----------
fc_2028 = (forecast_intervals
           .query("year == 2028 and scenario in ['LaNina','Neutral','ElNino']")
           .pivot(index="tower_id", columns="scenario", values="density_p50")
           .reset_index()
           .rename(columns={"LaNina":"gfd_LaNina","Neutral":"gfd_Neutral","ElNino":"gfd_ElNino"}))
fc_2028["dGFD_dENSO"] = (fc_2028["gfd_ElNino"] - fc_2028["gfd_LaNina"]) / 2.0
fc_2028["units"] = "flashes/km^2/yr per unit Niño 3.4 (year 2028)"
sensitivity = fc_2028.merge(spatial[["tower_id","lat","lng","elev_m"]], on="tower_id")
sensitivity.to_csv(OUT_DIR / "sensitivity.csv", index=False)
print(f"Wrote {OUT_DIR / 'sensitivity.csv'}")

print("\n=== All deliverables written ===")
for f in sorted(OUT_DIR.iterdir()):
    if f.is_file():
        print(f"  {f.name}  ({f.stat().st_size:,} bytes)")

# ---------- Headline forecast summary (GFD-centric) ----------
print("\n" + "=" * 72)
print("HEADLINE 2026–2030 PER-TOWER FORECAST")
print("=" * 72)

# Two scales — be explicit which we're showing.
hist_gfd_agg = line_year.set_index("year")["density"]
hist_density_panel = panel_m.groupby("year")["density"].mean()
ratio_check = hist_density_panel / hist_gfd_agg
print("\nHistorical density on two scales (DO NOT CONFLATE):")
print("  Year |  IEEE 1410 GFD  |  Panel-mean density  |  ratio")
print("       |  (agg row, unique-flash)  | (per-tower overlap-counted) |")
print("  " + "-" * 70)
for y in sorted(hist_gfd_agg.index):
    print(f"  {y}  |       {hist_gfd_agg[y]:.2f}        |        {hist_density_panel[y]:.2f}        |  {ratio_check[y]:.2f}")
print(f"  mean |       {hist_gfd_agg.mean():.2f}        |        {hist_density_panel.mean():.2f}        |  {ratio_check.mean():.2f}")
print("  (ratio ~5 = avg number of tower collection-circles each strike falls in)")

print("\nForecast values below are on the PANEL-MEAN (overlap-counted) scale —")
print("the same scale as the per-tower `density` column the user has. To compare")
print("to published IEEE 1410 GFD climatologies, divide by ~5.")
print("\nForecast mean per-tower density across all 281 towers (overlap-counted):")
print("  scenario     |  2026   2027   2028   2029   2030  |  5-yr mean")
print("  " + "-" * 65)
for sc in ["LaNina", "Neutral", "ElNino", "Marginalized"]:
    s = (forecast_intervals.query("scenario == @sc")
                           .groupby("year")["density_p50"]
                           .mean())
    five_yr = s.mean()
    yr_str = "  ".join(f"{v:>5.2f}" for v in s.values)
    print(f"  {sc:12s} |  {yr_str}  |  {five_yr:>8.2f}")

print("\nTop 5 hottest towers (5-year mean per-tower density, Neutral scenario):")
print("  (panel-scale; IEEE-1410-equivalent shown in parentheses ≈ /5)")
top5 = (forecast_intervals.query("scenario == 'Neutral'")
        .groupby("tower_id")["density_p50"].mean()
        .sort_values(ascending=False).head(5)
        .reset_index()
        .merge(spatial[["tower_id","lat","lng","elev_m"]], on="tower_id"))
for _, r in top5.iterrows():
    print(f"  Tower {int(r['tower_id']):>3d}  "
          f"lat={r['lat']:+.4f}  lng={r['lng']:+.4f}  "
          f"elev={r['elev_m']:>4.0f} m  "
          f"density = {r['density_p50']:>5.2f}  "
          f"(IEEE 1410 ≈ {r['density_p50']/5:.2f}) flashes/km²/yr")

print("\nClimate-scenario sensitivity (mean-GFD Δ vs Neutral, year 2028):")
sc_2028 = (forecast_intervals.query("year == 2028")
           .groupby("scenario")["density_p50"].mean())
neutral = sc_2028.get("Neutral", 0)
for sc in ["LaNina", "Neutral", "ElNino"]:
    delta = sc_2028.get(sc, 0) - neutral
    pct = 100 * delta / max(abs(neutral), 1e-6)
    print(f"  {sc:8s}: {sc_2028.get(sc, 0):.2f}  "
          f"({delta:+.2f} = {pct:+.1f}% vs Neutral)")

# CV honest reporting alongside.
print("\nCV skill (mean of folds — small absolute gain at n=7 years):")
skill_count = cv_summary.xs(("LOYO","count"), level=("scheme","target")).round(3)
skill_dens  = cv_summary.xs(("LOYO","density"), level=("scheme","target")).round(3)
print(f"  Count   LOYO RMSE  A={skill_count.loc['A','RMSE']:.3f}  "
      f"C={skill_count.loc['C','RMSE']:.3f}  "
      f"(C is {100*(1 - skill_count.loc['C','RMSE']/skill_count.loc['A','RMSE']):.1f}% better)")
print(f"  Density LOYO RMSE  A={skill_dens.loc['A','RMSE']:.3f}  "
      f"C={skill_dens.loc['C','RMSE']:.3f}  "
      f"(C is {100*(1 - skill_dens.loc['C','RMSE']/skill_dens.loc['A','RMSE']):.1f}% better)")
print("  → Model C beats climatology only marginally. Treat the climate-")
print("    scenario fan as a sensitivity analysis, not a skillful forecast.")
print(f"\n2025 hold-out check: Model C trained on 2019..2024, predicted 2025.")
print(f"  Pearson r between predicted and observed per-tower count = +0.237.")
print(f"  Below the 0.6 verification target — per-tower allocation noisy.")

print("\n" + "=" * 72)
print("Reproducibility: same RNG_SEED → identical numbers. Cached fetches in")
print("cache/ remain valid offline. See README sheet in the Excel for full")
print("units / caveats / dropped-model documentation.")